# Lab 20.3: Comprehensiveness and Sufficiency for a Sentiment Classifier

*Explainable AI in Practice*, Chapter 20. Run-only notebook: run the cells in order. Nothing needs to be edited.

You will explain the predictions of a small open sentiment classifier (DistilBERT fine-tuned on SST-2) with five token attribution methods and score each method with the ERASER measures of DeYoung et al. (2020):

* **Comprehensiveness**: how much the predicted-class probability drops when the tokens a method marks as most important are removed. Higher is better.
* **Sufficiency**: how much it drops when *only* those tokens are kept. Lower (closer to zero) is better.

Both are averaged over five sizes of the removed set (1, 5, 10, 20 and 50 percent of the tokens), the area over the perturbation curve. Every score is computed twice: once by **deleting** tokens from the sentence and once by **replacing** them with the `[MASK]` token. The forty review sentences are original, written for this lab.

Environment: the language model environment (`env/requirements-llm.txt`), kernel `xai-llm-venv`. The model (about 270 MB) is downloaded from the Hugging Face hub on first use.

In [1]:
import time, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import transformers
torch.manual_seed(0)
NAME = "distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForSequenceClassification.from_pretrained(NAME, attn_implementation="eager").eval()
print("torch", torch.__version__, "| transformers", transformers.__version__)

torch 2.6.0 | transformers 4.57.6


## Part A: forty sentences and the model's predictions

In [2]:
SENTENCES = [
 "The plot was predictable but the acting was wonderful.",
 "I loved every minute of this charming little film.",
 "A dull, lifeless script that wastes a talented cast.",
 "The soundtrack is beautiful and the photography is stunning.",
 "It tries hard to be clever and fails at almost everything.",
 "Not a single scene felt honest or earned.",
 "A warm, funny and surprisingly moving story.",
 "The second half drags, and the ending makes no sense.",
 "Brilliant performances lift a fairly ordinary script.",
 "I wanted to like it, but it was simply boring.",
 "The jokes land, the pacing is tight, and the cast is perfect.",
 "Messy, loud and far too long.",
 "An honest and gentle film about growing older.",
 "The dialogue is clumsy and the characters are paper thin.",
 "A delightful surprise from start to finish.",
 "Even the talented lead cannot save this mess.",
 "Smart, tender and beautifully shot.",
 "The special effects are cheap and the story is worse.",
 "It is not perfect, but it is never less than entertaining.",
 "A tedious exercise in style over substance.",
 "One of the most moving films I have seen this year.",
 "The director seems to have forgotten what the story was about.",
 "Fresh, bold and full of life.",
 "I left the theater bored and annoyed.",
 "The cast clearly had fun, and so did I.",
 "A confused and joyless sequel.",
 "Quietly powerful and wonderfully acted.",
 "Nothing about this film works.",
 "A clever script with real heart.",
 "The film is badly edited and painfully slow.",
 "Funny, sharp and never cruel.",
 "An ugly, cynical and empty film.",
 "The kind of movie that makes you want to call your family.",
 "Every twist is obvious from the first ten minutes.",
 "A gorgeous and haunting piece of work.",
 "The worst film I have seen in a long time.",
 "Charming performances and a lovely score.",
 "Flat jokes and a story that goes nowhere.",
 "A small film with a big, generous heart.",
 "Pretentious, humorless and dull.",
]

@torch.no_grad()
def prob(ids):
    return torch.softmax(model(input_ids=ids).logits, -1)

preds = [prob(tok(s, return_tensors="pt")["input_ids"])[0] for s in SENTENCES]
labels = [int(p.argmax()) for p in preds]
print("positive predictions:", sum(labels), "of", len(labels))
print("mean confidence:", round(float(np.mean([float(p.max()) for p in preds])), 4))

positive predictions: 20 of 40
mean confidence: 0.9997


## Part B: five attribution methods

* **Attention**: last layer, the `[CLS]` row, averaged over heads.
* **Gradient x input**: gradient of the predicted class's logit margin with respect to each token embedding, times the embedding, summed over the embedding dimensions.
* **Integrated Gradients**: logit margin, a zero-embedding baseline, 32 steps.
* **Occlusion**: drop in predicted-class probability when one token is replaced by `[MASK]`.

A **random** score is added as the baseline every method must beat.

In [3]:
def attributions(text, ig_steps=32):
    enc = tok(text, return_tensors="pt"); ids = enc["input_ids"]
    with torch.no_grad():
        out = model(**enc, output_attentions=True)
    p = torch.softmax(out.logits, -1)[0]; c = int(p.argmax()); n = ids.shape[1]
    res = {"Attention": out.attentions[-1][0, :, 0, :].mean(0).numpy()}
    E = model.distilbert.embeddings.word_embeddings(ids).detach()
    def margin(e):
        lg = model(inputs_embeds=e, attention_mask=enc["attention_mask"]).logits[0]
        return lg[c] - lg[1 - c]
    e = E.clone().requires_grad_(True); margin(e).backward()
    res["Gradient x input"] = (e.grad * E).sum(-1)[0].numpy()
    tot = torch.zeros_like(E)
    for a in (torch.arange(ig_steps) + 0.5) / ig_steps:
        ea = (a * E).clone().requires_grad_(True); margin(ea).backward(); tot += ea.grad
    res["Integrated Gradients"] = ((tot / ig_steps) * E).sum(-1)[0].detach().numpy()
    occ = np.zeros(n)
    for i in range(1, n - 1):
        ids2 = ids.clone(); ids2[0, i] = tok.mask_token_id
        occ[i] = float(p[c] - prob(ids2)[0, c])
    res["Occlusion"] = occ
    return ids, c, float(p[c]), res

ids, c, pc, res = attributions(SENTENCES[0])
toks = tok.convert_ids_to_tokens(ids[0])
print("sentence 1, predicted class", c, "probability", round(pc, 4))
for name, a in res.items():
    top = [toks[i] for i in np.argsort(-a[1:-1])[:3] + 1]
    print(f"{name:22s} top 3: {top}")

sentence 1, predicted class 1 probability 0.9998
Attention              top 3: ['wonderful', '.', 'acting']
Gradient x input       top 3: ['acting', 'was', 'but']
Integrated Gradients   top 3: ['wonderful', 'acting', 'but']
Occlusion              top 3: ['wonderful', 'but', '.']


**Checkpoint 1.** For the first sentence the model is almost certain (probability 0.9998). Integrated Gradients and occlusion put "wonderful" first; attention puts it first too, followed by the full stop; gradient x input puts "acting", "was" and "but" first. Which ranking looks plausible is not yet evidence: Part C tests them.

## Part C: comprehensiveness and sufficiency, two ways of removing tokens

In [4]:
def remove(ids, drop, how):
    if how == "mask":
        ids2 = ids.clone(); ids2[0, list(drop)] = tok.mask_token_id; return ids2
    keep = [i for i in range(ids.shape[1]) if i not in set(drop)]
    return ids[:, keep]

def comp_suff(ids, c, attr, how, fracs=(0.01, 0.05, 0.1, 0.2, 0.5)):
    n = ids.shape[1]; inner = np.arange(1, n - 1)      # never remove [CLS] or [SEP]
    order = inner[np.argsort(-attr[inner])]
    p0 = float(prob(ids)[0, c]); comp, suff = [], []
    for fr in fracs:
        k = max(1, int(round(fr * len(inner))))
        top = order[:k]; rest = [i for i in inner if i not in set(top)]
        comp.append(p0 - float(prob(remove(ids, top, how))[0, c]))
        suff.append(p0 - float(prob(remove(ids, rest, how))[0, c]))
    return float(np.mean(comp)), float(np.mean(suff))

t0 = time.time()
METHODS = ["Attention", "Gradient x input", "Integrated Gradients", "Occlusion", "Random"]
rng = np.random.default_rng(0)
scores = {(m, h): [] for m in METHODS for h in ("delete", "mask")}
for s in SENTENCES:
    ids, c, pc, res = attributions(s)
    res["Random"] = rng.random(ids.shape[1])
    for m in METHODS:
        for h in ("delete", "mask"):
            scores[(m, h)].append(comp_suff(ids, c, res[m], h))
print(f"computed in {time.time() - t0:.0f} seconds")
print(f"{'method':22s} {'comp(del)':>9s} {'suff(del)':>9s} {'comp(mask)':>10s} {'suff(mask)':>10s}")
for m in METHODS:
    d = np.array(scores[(m, "delete")]); k = np.array(scores[(m, "mask")])
    print(f"{m:22s} {d[:,0].mean():9.3f} {d[:,1].mean():9.3f} {k[:,0].mean():10.3f} {k[:,1].mean():10.3f}")

computed in 265 seconds
method                 comp(del) suff(del) comp(mask) suff(mask)
Attention                  0.242     0.105      0.201      0.209
Gradient x input           0.034     0.306      0.029      0.348
Integrated Gradients       0.256     0.072      0.246      0.075
Occlusion                  0.249     0.053      0.238      0.073
Random                     0.069     0.266      0.052      0.292


**Checkpoint 2.** Compare every method with the random row. Which methods beat it on both measures under both removal styles? Does any method look better under one removal style than under the other? Note that occlusion was itself computed with `[MASK]`, so the mask-based score shares its assumption.

## Part D: is the ranking the same for every sentence?

In [5]:
for h in ("delete", "mask"):
    best = []
    for i in range(len(SENTENCES)):
        vals = {m: scores[(m, h)][i][0] for m in METHODS if m != "Random"}
        best.append(max(vals, key=vals.get))
    names, counts = np.unique(best, return_counts=True)
    print(h, "- method with the highest comprehensiveness, by sentence:", dict(zip(names, counts.tolist())))

delete - method with the highest comprehensiveness, by sentence: {'Attention': 3, 'Gradient x input': 4, 'Integrated Gradients': 14, 'Occlusion': 19}
mask - method with the highest comprehensiveness, by sentence: {'Attention': 4, 'Gradient x input': 4, 'Integrated Gradients': 14, 'Occlusion': 18}


**Checkpoint 3.** The average hides variation: no method is best on every sentence.

## Questions

1. Which method would you trust to tell a product team which words drove the classifier's decisions, and why? Cite both measures.
2. Gradient x input scores below the random baseline here. Integrated Gradients uses the same gradients, averaged along a path. Explain in your own words why a single gradient at the input can mislead when the model's output is saturated (probability close to 1).
3. Attention is often dismissed as an explanation (Chapter 14). How does it do on these measures, and what do the measures not tell you about it?
4. Deleting tokens and masking them give different numbers. Which removal style is closer to the inputs the model saw in training, and why does that matter?
5. Write two sentences for an evaluation report that state the result for the best method with its settings and caveats.